# Notebook 01 — Construcción del Stack Biestacional Sentinel-1 + Sentinel-2 (Desde Cero)

- **Instancia SEPAL recomendada:** `t2` o `m2` (0.04–0.10 USD/h). El cómputo pesado del mosaico corre en la nube de GEE.
- **Proyecciones:** Cada costa usa su zona UTM nativa a **10 m exactos** (`EPSG:32615` Pacífico, `EPSG:32616` Caribe) para garantizar píxeles cuadrados de 100 m², cero distorsión espacial y exactitud en filtros focales y cálculo de áreas en hectáreas.
- **Salidas:**
  - Stack Biestacional de **24 bandas predictoras** (10 S2 Seca + 10 S2 Lluvia + 2 S1 Seca + 2 S1 Lluvia) con doble anillo geométrico (`buffer 100 m` + `margen kernel 60 m`).
  - Imagen QA de **6 bandas** (`NOBS_S2_SECA`, `NOBS_S2_LLUVIA`, `NOBS_S1_SECA`, `NOBS_S1_LLUVIA`, `MASCARA_COMUN`, `MASCARA_AOI_BASE`).
  - Inventario trazable de escenas S1 y metadatos completos de escenas S2/S1 (Asset + CSV en Drive).

In [ ]:
# 1. Configuracion de la corrida — elegir REGION: "PACIFICO" o "CARIBE"
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ee
import geemap
import pandas as pd

from src.config_utils import ensure_output_dirs, init_gee, load_config
from src.stack_builder import (
    build_biseasonal_stack,
    check_region_orbits,
    diagnose_stack,
    get_stack_asset_ids,
    start_stack_exports,
)

# >>> CAMBIAR AQUI ENTRE "PACIFICO" Y "CARIBE" <<<
REGION = "PACIFICO"

cfg = load_config()
dirs = ensure_output_dirs(cfg)
init_gee(cfg)

print(f"Region seleccionada : {REGION} ({cfg['regiones'][REGION]['nombre_largo']})")
print(f"CRS objetivo        : {cfg['regiones'][REGION]['crs']}")
print(f"Metodo nubes S2     : {cfg['s2']['metodo_nubes']}")
print(f"Filtro speckle S1   : {cfg['s1']['speckle_filter']} | Terreno: {cfg['s1']['terrain_model']}")
print(f"Assets destino      :", json.dumps(get_stack_asset_ids(REGION, cfg), indent=2))

In [ ]:
# 2. Diagnostico de orbitas Sentinel-1 (ASCENDING vs DESCENDING)
# Ejecutar siempre antes de fijar 'orbit_pass' en config.yaml (especialmente en CARIBE)
orbitas_info = check_region_orbits(REGION, cfg)
filas = []
for temp, data in orbitas_info.items():
    for orb, d in data.items():
        filas.append({
            "region": REGION,
            "temporada": temp,
            "orbita": orb,
            "n_escenas": d["n_escenas"],
            "orbitas_relativas": ", ".join(str(x) for x in d["orbitas_relativas"]),
        })
df_orbitas = pd.DataFrame(filas)
display(df_orbitas)
print("Orbita actualmente configurada en config.yaml:", cfg["regiones"][REGION]["orbit_pass"])

In [ ]:
# 3. Construir el Stack Biestacional (24 bandas) + QA (6 bandas) en Python GEE
bundle = build_biseasonal_stack(REGION, cfg)
print("Stack construido en grafo de GEE (evaluacion diferida lista para QA).")

In [ ]:
# 4. Control de calidad cuantitativo previo a exportar
diag = diagnose_stack(bundle, cfg, n_pixels=80)

print(f"=== DIAGNOSTICO STACK BIESTACIONAL {REGION} ===")
print(f"Bandas stack ({len(diag['bandas_stack'])}):", diag["bandas_stack"])
print(f"Bandas QA    ({len(diag['bandas_qa'])}):", diag["bandas_qa"])
print(f"S2 SECA   -> {diag['n_s2_seca']} escenas | Tiles: {diag['tiles_s2_seca']}")
print(f"S2 LLUVIA -> {diag['n_s2_lluvia']} escenas | Tiles: {diag['tiles_s2_lluvia']}")
print(f"S1 SECA   -> {diag['n_s1_seca']} escenas | Orbitas rel.: {diag['orbitas_rel_s1_seca']}")
print(f"S1 LLUVIA -> {diag['n_s1_lluvia']} escenas | Orbitas rel.: {diag['orbitas_rel_s1_lluvia']}")
print(f"Fraccion de pixeles validos (MASCARA_COMUN) dentro del AOI: {diag['fraccion_valida_aoi']*100:.2f}%")

stats_rows = []
for key in ["stats_VV_SECA", "stats_VH_SECA", "stats_VV_LLUVIA", "stats_VH_LLUVIA", "stats_B8_SECA", "stats_B8_LLUVIA"]:
    st = diag[key]
    stats_rows.append({
        "banda": key.replace("stats_", ""),
        "min": st.get("min"),
        "mean": st.get("mean"),
        "max": st.get("max"),
        "std": st.get("total_sd"),
    })
df_diag = pd.DataFrame(stats_rows)
display(df_diag)

# Guardar respaldo del diagnostico
out_diag = dirs["tablas"] / f"diagnostico_stack_{REGION.lower()}.csv"
df_diag.to_csv(out_diag, index=False)
print(f"Diagnostico guardado en: {out_diag}")

In [ ]:
# 5. Control de calidad visual interactivo (geemap)
stack = bundle["stack"]
qa = bundle["qa"]

m = geemap.Map()
m.centerObject(bundle["aoi_base"], 10)
m.add_basemap("SATELLITE")

m.addLayer(stack, {"bands": ["B4_SECA", "B3_SECA", "B2_SECA"], "min": 0, "max": 0.30}, "S2 Seca RGB", True)
m.addLayer(stack, {"bands": ["B8_SECA", "B11_SECA", "B4_SECA"], "min": 0, "max": 0.40}, "S2 Seca Falsocolor Manglar (B8-B11-B4)", False)
m.addLayer(stack, {"bands": ["B4_LLUVIA", "B3_LLUVIA", "B2_LLUVIA"], "min": 0, "max": 0.30}, "S2 Lluvia RGB", False)
m.addLayer(stack.select("VH_SECA"), {"min": -28, "max": -8, "palette": ["0015FF", "00FFFF", "FFFF00", "FF0000"]}, "SAR VH Seca (dB)", False)
m.addLayer(stack.select("VH_LLUVIA"), {"min": -28, "max": -8, "palette": ["0015FF", "00FFFF", "FFFF00", "FF0000"]}, "SAR VH Lluvia (dB)", False)
m.addLayer(qa.select("NOBS_S2_LLUVIA"), {"min": 0, "max": 12, "palette": ["FF0000", "FFFF00", "00AA00"]}, "QA NOBS S2 Lluvia", False)
m.addLayer(qa.select("MASCARA_AOI_BASE"), {"min": 0, "max": 1, "palette": ["FF8800", "0055FF"]}, "QA Nucleo (Azul) vs Buffer 100m (Naranja)", False)
m.addLayer(bundle["aoi_base"], {"color": "FFFFFF"}, f"AOI Base {REGION}", False)

# Superponer capa guia de manglar para referencia visual
try:
    guia = ee.Image(cfg["assets"]["mangle_guia"])
    m.addLayer(guia.select(0).selfMask(), {"min": 1, "max": 5, "palette": ["00FF66"]}, "Guia DinMangle 2020-2022", False)
except Exception:
    pass

m

In [ ]:
# 6. Lanzar exportaciones a GEE Assets y Drive
# Cambia EJECUTAR_EXPORTS = True solo cuando hayas revisado el QA visual y numerico.
EJECUTAR_EXPORTS = False

if EJECUTAR_EXPORTS:
    tasks = start_stack_exports(bundle, cfg, export_tables_to_drive=True)
    print("Tareas enviadas a Google Earth Engine:")
    print(json.dumps(tasks, indent=2))
else:
    print("Modo seguro activo (EJECUTAR_EXPORTS = False). No se enviaron tareas.")
    print("Cambia a True y vuelve a ejecutar esta celda cuando el QA este aprobado.")

In [ ]:
# 7. Monitorear el estado de las tareas activas en GEE
estados = []
for task in ee.batch.Task.list()[:10]:
    st = task.status()
    estados.append({
        "descripcion": st.get("description"),
        "estado": st.get("state"),
        "tipo": st.get("task_type"),
        "error": st.get("error_message", ""),
    })
display(pd.DataFrame(estados))